# Tracking Political Change with Embeddings of Parliamentary Speeches
***
# Aggregate Embeddings

In [ ]:
import pandas as pd
import numpy as np
from aggregation_functions.py import aggregate_by_mp_year, aggregate_by_party_year

In this notebook, we aggregate the speech-level embeddings to the MP-year and party-year level, separately for the off-the-shelf and the fine-tuned embeddings, so that both can later be compared using the same validation pipeline. We decided to aggregate by MP and year, as the average number of speeches per year and MP is 7.

Let's first check how many MPs gave speeches for different parties within the same year. Since it's the same set of speeches (only the embeddings differ), we can use either data set for this check (the one containing the off-the-shelf model's embeddings, or the one containing the fine-tuned model's).

In [2]:
data_offtheshelf = pd.read_parquet("jina_v3_offtheshelf_full.parquet")
data_offtheshelf.head(10)

,id,speechContent,politicianId,politicianName,firstName,lastName,positionShort,factionId,party,date,year,year_month,n_tokens,used_in_finetune,used_in_party_probe_val,used_in_party_probe_test,embedding
0,604683,Meine werten Kolleginnen und Kollegen! Liebe K...,11001434,Christoph Matschie,Christoph,Matschie,Member of Parliament,23,SPD,2000-01-19,2000,2000-01,1209,False,False,False,"[0.03406383, -0.103025556, 0.024277067, -0.025..."
1,604883,Rot-grün und Bildungsministerin Bulmahn sind w...,11002754,thomas rachel,thomas,rachel,Member of Parliament,4,CDU/CSU,2000-01-20,2000,2000-01,988,False,False,False,"[-0.016405942, -0.09249713, 0.033231184, -0.03..."
2,604885,Meine sehr verehrten Damen und Herren! Ich ste...,11000431,peter eckardt,peter,eckardt,Member of Parliament,23,SPD,2000-01-20,2000,2000-01,1087,False,False,False,"[-0.021692578, -0.10076961, 0.036127716, -0.02..."
3,604905,Die aktuelle Lage der deutschen Werftindustrie...,11001180,jürgen koppelin,jürgen,koppelin,Member of Parliament,13,FDP,2000-01-20,2000,2000-01,1550,False,False,False,"[0.030878019, -0.1255583, -0.027395273, -0.072..."
4,605067,Verehrte Frau Wehrbeauftragte! Liebe Kolleginn...,11003216,anita schäfer,anita,schäfer,Member of Parliament,4,CDU/CSU,2000-01-21,2000,2000-01,1552,False,False,False,"[-0.027387418, -0.12631616, 0.03779924, -0.017..."
5,605567,Sehr geehrte Kolleginnen und Kollegen! Verkehr...,11002457,Reinhard Weis,Reinhard,Weis,Member of Parliament,23,SPD,2000-01-27,2000,2000-01,1989,False,False,False,"[-0.00957919, -0.16544683, 0.022292798, -0.005..."
6,605569,"Meine sehr verehrten Damen und Herren! Das, wo...",11002651,Ulf Fink,Ulf,Fink,Member of Parliament,4,CDU/CSU,2000-01-27,2000,2000-01,1028,False,False,False,"[-0.09073022, -0.011530855, -0.07303492, 0.047..."
7,605685,In meinem Diskussionsbeitrag zum Berufsbildung...,11001033,rainer jork,rainer,jork,Member of Parliament,4,CDU/CSU,2000-01-28,2000,2000-01,1472,False,False,False,"[-0.002782963, -0.15656407, 0.016086992, -0.01..."
8,605717,Die Große Anfrage der F.D.P. greift die Situat...,11002720,steffi lemke,steffi,lemke,Member of Parliament,3,Grüne,2000-01-28,2000,2000-01,1550,False,False,False,"[0.008783339, -0.082546234, 0.0011761445, -0.0..."
9,606154,"Kollege Brüderle, als Sie in Ihrer Rede Walter...",11003195,Klaus Wolfgang Müller,Klaus Wolfgang,Müller,Member of Parliament,3,Grüne,2000-02-17,2000,2000-02,2428,False,False,False,"[0.030268304, -0.029429842, -0.00093157805, -0..."


In [3]:
party_switches = (
    data_offtheshelf.groupby(["politicianId", "year"])["party"]
    .nunique()
    .reset_index(name="n_parties")
)
party_switches = party_switches[party_switches["n_parties"] > 1]

print(party_switches)

      politicianId  year  n_parties
3008      11002813  2009          2
4211      11003206  2002          2
4214      11003206  2005          2


As this number is very low, we can neglect those cases and simply assign the party they belonged to when they gave their first speech in that year. The aggregation function we use can be found in the file 'aggregate_embeddings.py'. It also contains a function to aggregate by party and year as we need this level of aggregation for comparison with ideology scores. 

### Off-the-shelf embeddings

We first apply the aggregation functions to the off-the-shelf embeddings.

In [6]:
df_agg_offtheshelf = aggregate_by_mp_year(data_offtheshelf)
df_agg_offtheshelf.to_parquet("mp_year_embeddings_offtheshelf.parquet")

In [7]:
df_agg_by_party_offtheshelf = aggregate_by_party_year(data_offtheshelf)
df_agg_by_party_offtheshelf.to_parquet("party_year_embeddings_offtheshelf.parquet")

### Fine-tuned embeddings

We repeat the same aggregation on the fine-tuned embeddings, using the same two functions defined above.

In [8]:
data_finetuned = pd.read_parquet("jina_v3_contrastive_full.parquet")
data_finetuned.head(10)

,id,speechContent,politicianId,politicianName,firstName,lastName,positionShort,factionId,party,date,year,year_month,n_tokens,used_in_finetune,used_in_party_probe_val,used_in_party_probe_test,used_finetuned_weights,embedding
0,604683,Meine werten Kolleginnen und Kollegen! Liebe K...,11001434,Christoph Matschie,Christoph,Matschie,Member of Parliament,23,SPD,2000-01-19,2000,2000-01,1209,False,False,False,True,"[0.03820973, -0.11824905, 0.035603546, -0.0051..."
1,604883,Rot-grün und Bildungsministerin Bulmahn sind w...,11002754,thomas rachel,thomas,rachel,Member of Parliament,4,CDU/CSU,2000-01-20,2000,2000-01,988,False,False,False,True,"[0.043069527, -0.1234621, 0.08022803, 0.004297..."
2,604885,Meine sehr verehrten Damen und Herren! Ich ste...,11000431,peter eckardt,peter,eckardt,Member of Parliament,23,SPD,2000-01-20,2000,2000-01,1087,False,False,False,True,"[0.03223048, -0.10839531, 0.07345019, -0.00229..."
3,604905,Die aktuelle Lage der deutschen Werftindustrie...,11001180,jürgen koppelin,jürgen,koppelin,Member of Parliament,13,FDP,2000-01-20,2000,2000-01,1550,False,False,False,True,"[0.057155393, -0.12872227, 0.027934263, -0.038..."
4,605067,Verehrte Frau Wehrbeauftragte! Liebe Kolleginn...,11003216,anita schäfer,anita,schäfer,Member of Parliament,4,CDU/CSU,2000-01-21,2000,2000-01,1552,False,False,False,True,"[0.022645857, -0.14240317, 0.05878738, -0.0094..."
5,605567,Sehr geehrte Kolleginnen und Kollegen! Verkehr...,11002457,Reinhard Weis,Reinhard,Weis,Member of Parliament,23,SPD,2000-01-27,2000,2000-01,1989,False,False,False,True,"[0.018086893, -0.16677636, 0.035299327, 0.0012..."
6,605569,"Meine sehr verehrten Damen und Herren! Das, wo...",11002651,Ulf Fink,Ulf,Fink,Member of Parliament,4,CDU/CSU,2000-01-27,2000,2000-01,1028,False,False,False,True,"[-0.004370118, -0.03069762, -0.005006273, 0.04..."
7,605685,In meinem Diskussionsbeitrag zum Berufsbildung...,11001033,rainer jork,rainer,jork,Member of Parliament,4,CDU/CSU,2000-01-28,2000,2000-01,1472,False,False,False,True,"[0.046840094, -0.1519062, 0.042674746, 0.01096..."
8,605717,Die Große Anfrage der F.D.P. greift die Situat...,11002720,steffi lemke,steffi,lemke,Member of Parliament,3,Grüne,2000-01-28,2000,2000-01,1550,False,False,False,True,"[0.028807659, -0.104269415, 0.03252067, -0.010..."
9,606154,"Kollege Brüderle, als Sie in Ihrer Rede Walter...",11003195,Klaus Wolfgang Müller,Klaus Wolfgang,Müller,Member of Parliament,3,Grüne,2000-02-17,2000,2000-02,2428,False,False,False,True,"[0.044166, -0.046673443, 0.020151215, -0.01457..."


In [9]:
df_agg_finetuned = aggregate_by_mp_year(data_finetuned)
df_agg_finetuned.to_parquet("mp_year_embeddings_finetuned.parquet")

In [10]:
df_agg_by_party_finetuned = aggregate_by_party_year(data_finetuned)
df_agg_by_party_finetuned.to_parquet("party_year_embeddings_finetuned.parquet")